In [6]:
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool, ToolRuntime
from langchain_core.utils.uuid import uuid7
from langchain_openai import ChatOpenAI
import os

os.environ["DEEPSEEK_API_KEY"] = "sk-49892f84100b4c73843a82e371fc3522"

USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com",
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com",
    },
}


@dataclass
class UserContext:
    user_id: str


@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id

    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return (
            f"Account holder: {user['name']}\n"
            f"Type: {user['account_type']}\n"
            f"Balance: ${user['balance']}"
        )
    return "User not found"

agent = create_agent(
    model = "deepseek-flash",
    tools=[get_account_info],
    context_schema=UserContext,
    system_prompt="You are a financial assistant.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my current balance?"}]},
    config={"configurable": {"thread_id": str(uuid7())}},
    context=UserContext(user_id="user123"),
)
result

{'messages': [HumanMessage(content="What's my current balance?", additional_kwargs={}, response_metadata={}, id='cd9f1439-56a2-4713-a1f2-376cfa17aa58'),
  AIMessage(content="I'll look that up for you.", additional_kwargs={'refusal': None, 'reasoning_content': "The user is asking about their current balance. I have a tool `get_account_info` that gets the current user's account information. Let me call it."}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 287, 'total_tokens': 349, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 32, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 159}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9

In [7]:
[message.content for message in result["messages"]]

["What's my current balance?",
 "I'll look that up for you.",
 'Account holder: Alice Johnson\nType: Premium\nBalance: $5000',
 "Hi Alice! Here's your account summary:\n\n- **Account holder:** Alice Johnson\n- **Type:** Premium\n- **Balance:** $5,000\n\nIs there anything else you'd like to know about your account?"]